# Notebook 11 — LayoutLMv3 Data Preparation

Converts FATURA's pre-computed `layoutlm_HF_format` annotations into a
HuggingFace `DatasetDict` ready for LayoutLMv3 token-classification fine-tuning.

**Why no OCR here?**  
FATURA ships with `layoutlm_HF_format/` JSON files that already contain
tokenised `words`, pixel-coordinate `bboxes`, and integer `ner_tags`.  
Using these is faster and more accurate than re-running Tesseract on synthetic
images that were never designed as OCR inputs.  
*(Tesseract is still used in Notebook 13 for arbitrary real-world invoice images.)*

**Target extraction fields (6):**

| Field | FATURA tag ID(s) |
|---|---|
| INVOICE_NUMBER | 12 |
| INVOICE_DATE | 3 |
| DUE_DATE | 4 |
| ISSUER_NAME | 6 |
| RECIPIENT_NAME | 5, 8 |
| TOTAL_AMOUNT | 1 |
| O (outside) | 0, 2, 7, 9, 10, 11, 13 |

BIO scheme → 13 labels total (O + B/I × 6 fields).  
Split alignment: uses `train_fatura.csv`, `val_fatura.csv`, `test_fatura.csv`
invoice rows (not FATURA's own strat1 splits).

## 0. Install / check dependencies

In [7]:
import datasets

In [8]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import datasets
    print(f'datasets {datasets.__version__} already installed')
except ImportError:
    print('Installing datasets…')
    pip_install('datasets')
    import datasets
    print(f'datasets {datasets.__version__} installed')

datasets 4.5.0 already installed


## 1. Imports & paths

In [9]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import Dataset, DatasetDict, Features, Sequence, Value, ClassLabel

# ── Paths ────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()   # document-classification/
DATA_ROOT    = PROJECT_ROOT / 'data'

FATURA_ANN_DIR = (
    DATA_ROOT / 'raw' / 'fatura' / 'images'
    / 'invoices_dataset_final' / 'Annotations' / 'layoutlm_HF_format'
)
FATURA_IMG_DIR = (
    DATA_ROOT / 'raw' / 'fatura' / 'images'
    / 'invoices_dataset_final' / 'images'
)

CSV_DIR     = DATA_ROOT / 'processed'
OUT_DIR     = DATA_ROOT / 'processed' / 'layoutlmv3_dataset'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('FATURA annotation dir:', FATURA_ANN_DIR)
print('Output dir:', OUT_DIR)
print('Ann files available:', sum(1 for _ in FATURA_ANN_DIR.glob('*.json')))

FATURA annotation dir: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/fatura/images/invoices_dataset_final/Annotations/layoutlm_HF_format
Output dir: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/processed/layoutlmv3_dataset
Ann files available: 10000


## 2. Label schema

In [10]:
# ── 6 target fields ──────────────────────────────────────────────────────────
TARGET_FIELDS = [
    'INVOICE_NUMBER',   # FATURA tag 12
    'INVOICE_DATE',     # FATURA tag 3
    'DUE_DATE',         # FATURA tag 4
    'ISSUER_NAME',      # FATURA tag 6
    'RECIPIENT_NAME',   # FATURA tags 5, 8  (BUYER and BILL_TO)
    'TOTAL_AMOUNT',     # FATURA tag 1
]

# BIO labels: O, then B-/I- for each field
BIO_LABELS = ['O']
for f in TARGET_FIELDS:
    BIO_LABELS.append(f'B-{f}')
    BIO_LABELS.append(f'I-{f}')

label2id = {lbl: i for i, lbl in enumerate(BIO_LABELS)}
id2label = {i: lbl for i, lbl in enumerate(BIO_LABELS)}

print(f'Total BIO labels: {len(BIO_LABELS)}')
for i, lbl in id2label.items():
    print(f'  {i:2d}  {lbl}')

# Save for training / inference notebooks
with open(OUT_DIR / 'label2id.json', 'w') as f:
    json.dump(label2id, f, indent=2)
with open(OUT_DIR / 'id2label.json', 'w') as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)
print('\nSaved label2id.json and id2label.json')

Total BIO labels: 13
   0  O
   1  B-INVOICE_NUMBER
   2  I-INVOICE_NUMBER
   3  B-INVOICE_DATE
   4  I-INVOICE_DATE
   5  B-DUE_DATE
   6  I-DUE_DATE
   7  B-ISSUER_NAME
   8  I-ISSUER_NAME
   9  B-RECIPIENT_NAME
  10  I-RECIPIENT_NAME
  11  B-TOTAL_AMOUNT
  12  I-TOTAL_AMOUNT

Saved label2id.json and id2label.json


## 3. FATURA tag-ID → BIO label mapping

In [11]:
# Maps each FATURA integer tag ID to one of our 6 field names (or None → O).
# Derived by inspecting words in layoutlm_HF_format files:
#   1  → TOTAL_AMOUNT    (e.g. "TOTAL", "398.77", "$")
#   3  → INVOICE_DATE    (e.g. "Invoice", "Date", "06-Oct-2000")
#   4  → DUE_DATE        (e.g. "Due", "Date", "27-May-2019")
#   5  → RECIPIENT_NAME  (BUYER variant, e.g. "Buyer", "Chelsea", "Mata")
#   6  → ISSUER_NAME     (e.g. "Mclean-Cochran")
#   8  → RECIPIENT_NAME  (BILL_TO variant, e.g. "BILL_TO", "Timothy", "Scott")
#  12  → INVOICE_NUMBER  (e.g. "INVOICE", "#", "INV/09-99/237")
#  rest (0,2,7,9,10,11,13) → O

FATURA_TAG_TO_FIELD = {
    1:  'TOTAL_AMOUNT',
    3:  'INVOICE_DATE',
    4:  'DUE_DATE',
    5:  'RECIPIENT_NAME',
    6:  'ISSUER_NAME',
    8:  'RECIPIENT_NAME',
    12: 'INVOICE_NUMBER',
}

print('FATURA tag → field mapping:')
for tag_id in sorted(FATURA_TAG_TO_FIELD):
    print(f'  Tag {tag_id:2d} → {FATURA_TAG_TO_FIELD[tag_id]}')
print('  All other tag IDs → O')

FATURA tag → field mapping:
  Tag  1 → TOTAL_AMOUNT
  Tag  3 → INVOICE_DATE
  Tag  4 → DUE_DATE
  Tag  5 → RECIPIENT_NAME
  Tag  6 → ISSUER_NAME
  Tag  8 → RECIPIENT_NAME
  Tag 12 → INVOICE_NUMBER
  All other tag IDs → O


## 4. Conversion helper

In [12]:
# All FATURA images are 595 × 841 pixels.
# LayoutLMv3 expects bboxes normalised to [0, 1000].
IMG_W, IMG_H = 595, 841


def normalise_bbox(bbox, img_w=IMG_W, img_h=IMG_H):
    """Convert pixel [x0,y0,x1,y1] → LayoutLMv3 [0,1000] int coords."""
    x0, y0, x1, y1 = bbox
    x0 = max(0, min(int(x0 / img_w * 1000), 1000))
    y0 = max(0, min(int(y0 / img_h * 1000), 1000))
    x1 = max(0, min(int(x1 / img_w * 1000), 1000))
    y1 = max(0, min(int(y1 / img_h * 1000), 1000))
    # Ensure x0 <= x1 and y0 <= y1
    return [min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)]


def flat_tags_to_bio(fatura_tags):
    """
    Convert a list of flat FATURA entity-type IDs into BIO label IDs.

    FATURA annotations give the same integer tag to every token in an entity
    (no B/I distinction). We apply the BIO rule:
      - First token of a contiguous same-tag span → B-FIELD
      - Subsequent tokens of the same span      → I-FIELD
      - Tokens with unmapped tags               → O
    """
    bio_ids = []
    prev_field = None
    for tag in fatura_tags:
        field = FATURA_TAG_TO_FIELD.get(tag, None)
        if field is None:
            bio_ids.append(label2id['O'])
            prev_field = None
        elif field == prev_field:
            # Continuation of same entity span
            bio_ids.append(label2id[f'I-{field}'])
        else:
            # Start of a new entity span
            bio_ids.append(label2id[f'B-{field}'])
            prev_field = field
    return bio_ids


def load_annotation(ann_path, img_w=IMG_W, img_h=IMG_H):
    """
    Load one FATURA HF-format annotation and return a dict with:
      words, bboxes (normalised to [0,1000]), ner_tags (BIO label IDs).
    """
    with open(ann_path) as f:
        rec = json.load(f)

    words    = rec['words']
    bboxes   = [normalise_bbox(b, img_w, img_h) for b in rec['bboxes']]
    ner_tags = flat_tags_to_bio(rec['ner_tags'])

    return {'words': words, 'bboxes': bboxes, 'ner_tags': ner_tags}


# Quick sanity check — pick whichever annotation file exists for Instance49
import glob as _glob
_candidates = _glob.glob(str(FATURA_ANN_DIR / 'Template10_Instance49_hugg_*.json'))
assert _candidates, 'No annotation found for Template10_Instance49'
_sample = load_annotation(_candidates[0])
print('Sample words :', _sample['words'][:6])
print('Sample bboxes:', _sample['bboxes'][:3])
print('Sample labels:', [id2label[t] for t in _sample['ner_tags'][:12]])

Sample words : ['table', 'Buyer', 'Chelsea', 'Mata', '8355', 'James']
Sample bboxes: [[6, 454, 897, 561], [544, 197, 853, 297], [544, 197, 853, 297]]
Sample labels: ['O', 'B-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME']


## 5. Build split → stems mapping from project CSVs

In [13]:
def get_invoice_stems(split_name):
    """Return list of filename stems for invoice rows in a split CSV."""
    csv_path = CSV_DIR / f'{split_name}_fatura.csv'
    df = pd.read_csv(csv_path)
    inv = df[df['class_name'] == 'invoice'].copy()
    inv['stem'] = inv['file_path'].apply(lambda p: Path(p).stem)
    return inv['stem'].tolist()

train_stems = get_invoice_stems('train')
val_stems   = get_invoice_stems('val')
test_stems  = get_invoice_stems('test')

print(f'Invoice stems — train: {len(train_stems)}, val: {len(val_stems)}, test: {len(test_stems)}')
print('Example stem:', train_stems[0])

# Build stem → annotation-file lookup (glob once, index by stem)
ann_files = list(FATURA_ANN_DIR.glob('*.json'))
stem_to_ann = {}
for ann_file in ann_files:
    # Filename pattern: Template{N}_Instance{M}_hugg_{train|dev|test}.json
    # Stem of the image = everything before the last "_hugg_" suffix
    stem = re.sub(r'_hugg_(train|dev|test)$', '', ann_file.stem)
    stem_to_ann[stem] = ann_file

print(f'Annotation index built: {len(stem_to_ann)} entries')

# Check coverage
for split_name, stems in [('train', train_stems), ('val', val_stems), ('test', test_stems)]:
    missing = [s for s in stems if s not in stem_to_ann]
    print(f'{split_name}: {len(stems)} stems, missing annotations: {len(missing)}')

Invoice stems — train: 1734, val: 371, test: 372
Example stem: Template40_Instance176
Annotation index built: 10000 entries
train: 1734 stems, missing annotations: 0
val: 371 stems, missing annotations: 0
test: 372 stems, missing annotations: 0


## 6. Build HuggingFace DatasetDict

In [14]:
def build_split_records(stems, split_label):
    """
    Load all annotation files for a set of image stems and return
    a list of dicts with words, bboxes, ner_tags, image_path.
    """
    records = []
    skipped = 0
    for stem in stems:
        ann_path = stem_to_ann.get(stem)
        if ann_path is None:
            skipped += 1
            continue
        img_path = str(FATURA_IMG_DIR / f'{stem}.jpg')
        rec = load_annotation(ann_path)
        rec['image_path'] = img_path
        records.append(rec)
    if skipped:
        print(f'  [{split_label}] WARNING: {skipped} stems had no annotation file')
    return records


print('Building train split…')
train_records = build_split_records(train_stems, 'train')
print(f'  train: {len(train_records)} records')

print('Building val split…')
val_records = build_split_records(val_stems, 'val')
print(f'  val:   {len(val_records)} records')

print('Building test split…')
test_records = build_split_records(test_stems, 'test')
print(f'  test:  {len(test_records)} records')

Building train split…
  train: 1734 records
Building val split…
  val:   371 records
Building test split…
  test:  372 records


In [15]:
# Convert list-of-dicts → HuggingFace Dataset objects

def records_to_hf_dataset(records):
    """Convert list of record dicts to a HuggingFace Dataset."""
    # Transpose to column-oriented dict
    cols = {
        'image_path': [r['image_path'] for r in records],
        'words':      [r['words']      for r in records],
        'bboxes':     [r['bboxes']     for r in records],
        'ner_tags':   [r['ner_tags']   for r in records],
    }
    return Dataset.from_dict(cols)


dataset_dict = DatasetDict({
    'train': records_to_hf_dataset(train_records),
    'val':   records_to_hf_dataset(val_records),
    'test':  records_to_hf_dataset(test_records),
})

print(dataset_dict)
print('\nSample train record:')
sample = dataset_dict['train'][0]
print('  words    :', sample['words'][:6])
print('  bboxes   :', sample['bboxes'][:3])
print('  ner_tags :', [id2label[t] for t in sample['ner_tags'][:8]])
print('  img_path :', sample['image_path'])

DatasetDict({
    train: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 1734
    })
    val: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 371
    })
    test: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 372
    })
})

Sample train record:
  words    : ['table', 'Buyer', 'Courtney', 'Mills', '064', 'Long']
  bboxes   : [[16, 455, 924, 583], [23, 152, 326, 252], [23, 152, 326, 252]]
  ner_tags : ['O', 'B-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'I-RECIPIENT_NAME']
  img_path : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/raw/fatura/images/invoices_dataset_final/images/Template40_Instance176.jpg


## 7. Label distribution report

In [16]:
import pandas as pd

def label_distribution(dataset, split_name):
    counts = Counter()
    for rec in dataset:
        counts.update(rec['ner_tags'])
    rows = []
    for lid in range(len(BIO_LABELS)):
        rows.append({'label_id': lid, 'label': id2label[lid], 'count': counts.get(lid, 0)})
    df = pd.DataFrame(rows)
    df['pct'] = (df['count'] / df['count'].sum() * 100).round(2)
    return df

for split_name in ['train', 'val', 'test']:
    df_dist = label_distribution(dataset_dict[split_name], split_name)
    print(f'\n=== {split_name.upper()} LABEL DISTRIBUTION ===')
    print(df_dist.to_string(index=False))
    total_tokens = df_dist['count'].sum()
    entity_tokens = df_dist[df_dist['label'] != 'O']['count'].sum()
    print(f'Total tokens: {total_tokens:,}  |  Entity tokens: {entity_tokens:,}  ({entity_tokens/total_tokens*100:.1f}%)')


=== TRAIN LABEL DISTRIBUTION ===
 label_id            label  count   pct
        0                O  72141 61.81
        1 B-INVOICE_NUMBER   1515  1.30
        2 I-INVOICE_NUMBER   3030  2.60
        3   B-INVOICE_DATE   1701  1.46
        4   I-INVOICE_DATE   2585  2.21
        5       B-DUE_DATE    988  0.85
        6       I-DUE_DATE   1976  1.69
        7    B-ISSUER_NAME   1164  1.00
        8    I-ISSUER_NAME   1822  1.56
        9 B-RECIPIENT_NAME   1325  1.14
       10 I-RECIPIENT_NAME  24161 20.70
       11   B-TOTAL_AMOUNT   1380  1.18
       12   I-TOTAL_AMOUNT   2935  2.51
Total tokens: 116,723  |  Entity tokens: 44,582  (38.2%)

=== VAL LABEL DISTRIBUTION ===
 label_id            label  count   pct
        0                O  15320 61.03
        1 B-INVOICE_NUMBER    314  1.25
        2 I-INVOICE_NUMBER    628  2.50
        3   B-INVOICE_DATE    361  1.44
        4   I-INVOICE_DATE    547  2.18
        5       B-DUE_DATE    211  0.84
        6       I-DUE_DATE    422  1.

## 8. Save DatasetDict to disk

In [17]:
save_path = str(OUT_DIR)
dataset_dict.save_to_disk(save_path)
print(f'DatasetDict saved to: {save_path}')

# Verify round-trip
from datasets import load_from_disk
loaded = load_from_disk(save_path)
print('\nVerification — loaded DatasetDict:')
print(loaded)
print('\nSaved files:')
for p in sorted(OUT_DIR.iterdir()):
    print(' ', p.name)

Saving the dataset (1/1 shards): 100%|██████████| 372/372 [00:00<00:00, 57990.08 examples/s]

DatasetDict saved to: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/processed/layoutlmv3_dataset

Verification — loaded DatasetDict:
DatasetDict({
    train: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 1734
    })
    val: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 371
    })
    test: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 372
    })
})

Saved files:
  dataset_dict.json
  id2label.json
  label2id.json
  test
  train
  val


## 9. Summary

In [18]:
print('=' * 60)
print('NOTEBOOK 11 — DATA PREPARATION COMPLETE')
print('=' * 60)
print(f'Label schema  : {len(BIO_LABELS)} BIO labels')
print(f'              : {BIO_LABELS}')
print(f'Train samples : {len(dataset_dict["train"])}')
print(f'Val samples   : {len(dataset_dict["val"])}')
print(f'Test samples  : {len(dataset_dict["test"])}')
print(f'Output dir    : {OUT_DIR}')
print()
print('Artifacts written:')
print(f'  {OUT_DIR}/label2id.json')
print(f'  {OUT_DIR}/id2label.json')
print(f'  {OUT_DIR}/ (DatasetDict)')
print()
print('Next: Notebook 12 — LayoutLMv3 fine-tuning')

NOTEBOOK 11 — DATA PREPARATION COMPLETE
Label schema  : 13 BIO labels
              : ['O', 'B-INVOICE_NUMBER', 'I-INVOICE_NUMBER', 'B-INVOICE_DATE', 'I-INVOICE_DATE', 'B-DUE_DATE', 'I-DUE_DATE', 'B-ISSUER_NAME', 'I-ISSUER_NAME', 'B-RECIPIENT_NAME', 'I-RECIPIENT_NAME', 'B-TOTAL_AMOUNT', 'I-TOTAL_AMOUNT']
Train samples : 1734
Val samples   : 371
Test samples  : 372
Output dir    : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/processed/layoutlmv3_dataset

Artifacts written:
  /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/processed/layoutlmv3_dataset/label2id.json
  /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/processed/layoutlmv3_dataset/id2label.json
  /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/data/processed/layoutlmv3_dataset/ (DatasetDict)

Next: Notebook 12 — LayoutLMv3 fine